# Vitalidad Urbana v4 — D1 = mezcla de usos (espacial)
**TFM – Álvaro Fernández-Uribarri Poveda · QuEA 2025–26**

Versión derivada de la "limpia sin Airbnb/VUT". Cambio principal: **D1 ya no mide "economía de barrio"** (comercio cotidiano por keywords), sino la **mezcla de usos primarios** dentro de cada hexágono, medida de forma espacial.

Para cada local se mira la fracción de sus vecinos más cercanos que son de un **uso distinto** (comercio, hostelería, ocio, oficina, equipamiento) y se promedia por hexágono:
- usos entreverados (p. ej. oficinas dispersas entre comercio) → **alta** mezcla.
- usos en bloques separados o un solo uso (zona dormitorio / CBD puro) → **baja** mezcla.

Airbnb/VUT siguen fuera del índice (solo control/validación). Las celdas duplicadas del índice siguen desactivadas.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.iolib.summary2 import summary_col
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from scipy.spatial import cKDTree
from scipy import stats
from scipy.stats import rankdata, entropy as scipy_entropy
from shapely.geometry import Point
from pathlib import Path
import unicodedata, warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

BASE       = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA')
VITALIDAD  = BASE / 'Vitalidad'
CRS        = 'EPSG:25830'
SOL_X, SOL_Y = 440370, 4474018

GPKG_MORF    = BASE      / 'madrid_morfologia_h3_v10.gpkg'
LOCALES_CSV  = VITALIDAD / '200085-5-censo-locales.csv'
TERRAZAS_CSV = VITALIDAD / '200085-6-censo-locales.csv'
PARQUES_GEO  = VITALIDAD / '200761-1-parques-jardines-geo.geo'
MERCADOS_CSV = VITALIDAD / '200967-4-mercados-csv.csv'
ADRH_CSV     = VITALIDAD / '31097.csv'
CENSO21_GPKG = VITALIDAD / 'censo2021_seccen_madrid.gpkg'
CENTRAL_CSV  = VITALIDAD / 'centralidad_h3.csv'
AIRBNB_CSV   = VITALIDAD / 'airbnb_madrid_listings.csv'
VUT_SHP      = VITALIDAD / 'vut_shp' / 'VIVIENDAS_USO_TURISTICO' / 'VIVIENDAS_USO_TURISTICO.shp'

Path('imagenes_v4').mkdir(exist_ok=True)

print('Rutas:')
for p in [GPKG_MORF, LOCALES_CSV, TERRAZAS_CSV, PARQUES_GEO, MERCADOS_CSV,
          ADRH_CSV, CENSO21_GPKG, CENTRAL_CSV, AIRBNB_CSV, VUT_SHP]:
    print(f'  {p.name:<40} {"OK" if p.exists() else "FALTA"}')

## 1. Hexágonos H3

In [ ]:
gdf = gpd.read_file(str(GPKG_MORF)).to_crs(CRS)
gdf['area_km2'] = gdf.geometry.area / 1e6
gdf['centroid'] = gdf.geometry.centroid
gdf['dist_centro_km'] = np.sqrt((gdf['centroid'].x - SOL_X)**2 +
                                 (gdf['centroid'].y - SOL_Y)**2) / 1000
print(f'Hexagonos: {len(gdf)}')
print(gdf['tipologia_label'].value_counts())

## 2. D1 — Mezcla de usos (espacial)

D1 mide la **diversidad funcional de grano fino**: si dentro del hexágono conviven y están
entreverados distintos usos (comercio, hostelería, ocio, oficinas, equipamiento) la mezcla es
alta; si hay un solo uso o están segregados en bloques, es baja. Se calcula por vecindad
(heterogeneidad de los vecinos más cercanos de cada local).

In [ ]:
print('Cargando Censo de Locales...')
loc = pd.read_csv(LOCALES_CSV, sep=';',
    usecols=['coordenada_x_local','coordenada_y_local','id_situacion_local',
             'desc_division','desc_epigrafe'], low_memory=False)
loc_act = (loc[loc['id_situacion_local']==1]
           .dropna(subset=['coordenada_x_local','coordenada_y_local']).copy())
print(f'Activos con coords: {len(loc_act):,}')

# - Filtro jacobsiano (se mantiene: lo usan la celda 7 y la 8) -
DIVISIONES_JACOBS = {
    'COMERCIO AL POR MENOR, EXCEPTO DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS',
    'SERVICIOS DE COMIDAS Y BEBIDAS', 'OTROS SERVICIOS PERSONALES',
    'ACTIVIDADES DEPORTIVAS, RECREATIVAS Y DE ENTRETENIMIENTO',
    'REPARACIÓN DE ORDENADORES, EFECTOS PERSONALES Y ARTÍCULOS DE USO DOMÉSTICO',
    'ACTIVIDADES DE JUEGOS DE AZAR Y APUESTAS',
    'ACTIVIDADES DE AGENCIAS DE VIAJES, OPERADORES TURÍSTICOS, SERVICIOS DE RESERVAS Y ACTIVIDADES RELACIONADAS CON LOS MISMOS',
    'ACTIVIDADES DE CREACIÓN, ARTÍSTICAS Y ESPECTÁCULOS', 'ACTIVIDADES VETERINARIAS',
    'ACTIVIDADES DE BIBLIOTECAS, ARCHIVOS, MUSEOS Y OTRAS ACTIVIDADES CULTURALES',
    'ACTIVIDADES CINEMATOGRÁFICAS, DE VÍDEO Y DE PROGRAMAS DE TELEVISIÓN, GRABACIÓN DE SONIDO Y EDICIÓN MUSICAL',
    'VENTA Y REPARACIÓN DE VEHÍCULOS DE MOTOR Y MOTOCICLETAS', 'SERVICIOS DE ALOJAMIENTO',
    'ACTIVIDADES SANITARIAS', 'EDUCACIÓN', 'ACTIVIDADES ASOCIATIVAS',
    'ACTIVIDADES DE SERVICIOS SOCIALES SIN ALOJAMIENTO',
}
def norm(s):
    if pd.isna(s): return ''
    return unicodedata.normalize('NFC', str(s).upper().strip())
DIV_NORM = {norm(d) for d in DIVISIONES_JACOBS}
loc_act['_div'] = loc_act['desc_division'].apply(norm)
jacobs = loc_act[loc_act['_div'].isin(DIV_NORM)].copy()
jacobs['es_hosteleria'] = jacobs['_div'] == norm('SERVICIOS DE COMIDAS Y BEBIDAS')
print(f'Jacobianos: {len(jacobs):,}')

# - NUEVO: clasificación de USOS PRIMARIOS para medir MEZCLA (incluye OFICINAS) -
def _strip(s):
    s = unicodedata.normalize('NFD', str(s).upper())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

OFICINA_KW  = ['PROFESIONALES, CIENTIFICAS','JURIDICAS','CONTABILIDAD','SEDES CENTRALES',
               'CONSULTORIA','ARQUITECTURA E INGENIERIA','INVESTIGACION Y DESARROLLO',
               'PUBLICIDAD','FINANCIEROS','SEGUROS','INMOBILIARIAS','PROGRAMACION',
               'INFORMATICA','SERVICIOS DE INFORMACION','TELECOMUNICACIONES','EDICION',
               'ADMINISTRATIVAS','RELACIONADAS CON EL EMPLEO','ADMINISTRACION PUBLICA']
HOST_KW     = ['SERVICIOS DE COMIDAS Y BEBIDAS','SERVICIOS DE ALOJAMIENTO']
OCIO_KW     = ['DEPORTIVAS, RECREATIVAS','CREACION, ARTISTICAS','BIBLIOTECAS',
               'CINEMATOGRAFICAS','JUEGOS DE AZAR']
COMERCIO_KW = ['COMERCIO AL POR MENOR','VENTA Y REPARACION DE VEHICULOS']
EQUIP_KW    = ['SANITARIAS','EDUCACION','SERVICIOS SOCIALES','ASOCIATIVAS',
               'OTROS SERVICIOS PERSONALES','VETERINARIAS','REPARACION DE ORDENADORES']

def clasifica_uso(div):
    d = _strip(div)
    if any(k in d for k in OFICINA_KW):  return 'oficina'
    if any(k in d for k in HOST_KW):     return 'hosteleria'
    if any(k in d for k in OCIO_KW):     return 'ocio'
    if any(k in d for k in COMERCIO_KW): return 'comercio'
    if any(k in d for k in EQUIP_KW):    return 'equipamiento'
    return None   # industria, mayorista, transporte... fuera de la mezcla urbana

loc_act['uso'] = loc_act['desc_division'].apply(clasifica_uso)
usos = loc_act.dropna(subset=['uso']).copy()
print('\nLocales por USO (revisa que oficinas tenga sentido):')
print(usos['uso'].value_counts())

In [ ]:
# ── Clasificación ECONOMÍA DE BARRIO (vida cotidiana, no turística) ──────────
BARRIO_KW = [
    # Alimentación de proximidad
    'FRUTA','VERDURA','HORTALIZAS','PAN ','PANADERIA','PASTELERIA','CONFITERIA','REPOSTERIA',
    'CARNE','CARNICERIA','SALCHICHERIA','CHARCUTERIA','PESCADO','PESCADERIA','MARISCO',
    'AUTOSERVICIO','SUPERMERCADO','ULTRAMARINOS','HUEVOS','LACTEOS','FRUTOS SECOS',
    'PRODUCTOS ALIMENTICIOS','HERBOLARIO','VINO',
    # Cuidado personal recurrente
    'PELUQUERIA','BARBERIA','ESTETICA',
    # Salud de proximidad
    'FARMACIA','OPTICA','ORTOPEDIA','FISIOTERAPEUTA','PARAFARMACIA',
    # Mantenimiento del hogar
    'FERRETERIA','DROGUERIA','PERFUMERIA','CERRAJERIA','CERRAJERO','MERCERIA',
    'REPARACION DE CALZADO','REPARACION DE ELECTRODOMESTICOS','ARREGLOS',
    # Gestiones diarias
    'TABACO','ESTANCO','PAPELERIA','PERIODICOS','REVISTAS','PRENSA',
    'TINTORERIA','LAVANDERIA',
    # Otros de proximidad
    'FLORISTERIA','FLORES','CALZADO','ZAPATERIA','RELOJERIA','JOYERIA',
]
# Normalizar texto sin tildes para comparación robusta
def _na(s):
    s = unicodedata.normalize('NFD', s.upper())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')
BARRIO_KW_NOACC = [_na(k) for k in BARRIO_KW]
def es_barrio(ep):
    if pd.isna(ep): return False
    e = _na(str(ep))
    return any(k in e for k in BARRIO_KW_NOACC)

jacobs['es_barrio'] = jacobs['desc_epigrafe'].apply(es_barrio)
print(f'Locales economía de barrio: {jacobs["es_barrio"].sum():,} '
      f'({jacobs["es_barrio"].mean()*100:.1f}% de jacobianos)')
print(f'Locales hostelería:         {jacobs["es_hosteleria"].sum():,} '
      f'({jacobs["es_hosteleria"].mean()*100:.1f}% de jacobianos)')

In [ ]:
import geopandas as gpd
from scipy.spatial import cKDTree
from scipy.stats import entropy as scipy_entropy

# ── D1 = MEZCLA DE USOS (entropía: cuánta variedad de usos hay) ──────────────
from scipy.stats import entropy as scipy_entropy

gusos = gpd.GeoDataFrame(usos, geometry=gpd.points_from_xy(
        usos['coordenada_x_local'], usos['coordenada_y_local']), crs=CRS)
ju = gpd.sjoin(gusos[['uso','geometry']], gdf[['hex_id','geometry']],
               how='inner', predicate='within')

MIN_LOCALES = 5
N_USOS = 5   # comercio, hostelería, ocio, oficina, equipamiento

grp = ju.groupby('hex_id')['uso']
agg = pd.DataFrame({
    'n_locales_uso': grp.size(),
    'n_usos':        grp.nunique(),
    'mix_usos':      grp.agg(lambda s: float(scipy_entropy(s.value_counts()) / np.log(N_USOS))),
}).reset_index()
agg.loc[agg['n_locales_uso'] < MIN_LOCALES, 'mix_usos'] = np.nan   # pocos negocios = no fiable

d1 = d1.merge(agg[['hex_id','mix_usos','n_usos','n_locales_uso']], on='hex_id', how='left')
print('D1 — mezcla de usos (entropía 0..1, 1 = todos los usos en igual proporción):')
print(d1[['mix_usos','n_usos','n_locales_uso']].describe().round(3))

In [ ]:
import geopandas as gpd, numpy as np, matplotlib.pyplot as plt
import osmnx as ox
from pathlib import Path

# Geometría hexagonal + la dimensión
g = gpd.read_file(
    r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\madrid_morfologia_h3_v10.gpkg'
).to_crs(25830)
gm = g.merge(m[['hex_id','dim1']], on='hex_id', how='left')

# Separar urbano / no urbano (gris)
built = gm['cat_n_buildings'].fillna(0) >= 5
disp, grey = gm[built].copy(), gm[~built].copy()

fig, ax = plt.subplots(figsize=(12, 12))
vmin, vmax = np.nanpercentile(disp['dim1'].astype(float).dropna(), (2, 98))
grey.plot(ax=ax, color='#E8E8E8', linewidth=0)
disp.plot(column='dim1', ax=ax, cmap='magma', vmin=vmin, vmax=vmax, linewidth=0,
          legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30},
          missing_kwds={'color':'#E8E8E8'})

# --- Distritos ---
ox.settings.use_cache = True
try:
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level': '9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(25830)
    dist.boundary.plot(ax=ax, color='black', linewidth=0.9, alpha=0.65)
    for _, r in dist.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(r['name'], (c.x, c.y), fontsize=6.5, ha='center', va='center',
                    fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
    print('Distritos cargados:', len(dist))
except Exception as e:
    print('sin distritos (¿internet/osmnx?):', e)

ax.set_title('Dim 1 · Mezcla de usos · gris = no urbano excluido', fontsize=12)
ax.set_axis_off(); ax.set_aspect('equal')
plt.tight_layout()
Path('imagenes').mkdir(exist_ok=True)
plt.savefig('imagenes/mapa_dim1_mezcla_usos.png', dpi=170, bbox_inches='tight')
plt.show()

In [ ]:
import geopandas as gpd, numpy as np, pandas as pd, unicodedata
from scipy.spatial import cKDTree

# 1) Locales activos clasificados en USO + epígrafe detallado
loc = pd.read_csv(LOCALES_CSV, sep=';',
    usecols=['coordenada_x_local','coordenada_y_local','id_situacion_local','desc_division','desc_epigrafe'],
    low_memory=False)
la = loc[loc['id_situacion_local']==1].dropna(subset=['coordenada_x_local','coordenada_y_local']).copy()

def _na(s):
    s = unicodedata.normalize('NFD', str(s).upper())
    return ''.join(c for c in s if unicodedata.category(c) != 'Mn')

USOS_KW = {
 'oficina':      ['PROFESIONALES, CIENTIFICAS','JURIDICAS','CONTABILIDAD','SEDES CENTRALES','CONSULTORIA',
                  'ARQUITECTURA E INGENIERIA','INVESTIGACION Y DESARROLLO','PUBLICIDAD','FINANCIEROS','SEGUROS',
                  'INMOBILIARIAS','PROGRAMACION','INFORMATICA','SERVICIOS DE INFORMACION','TELECOMUNICACIONES',
                  'EDICION','ADMINISTRATIVAS','RELACIONADAS CON EL EMPLEO','ADMINISTRACION PUBLICA'],
 'hosteleria':   ['SERVICIOS DE COMIDAS Y BEBIDAS','SERVICIOS DE ALOJAMIENTO'],
 'ocio':         ['DEPORTIVAS, RECREATIVAS','CREACION, ARTISTICAS','BIBLIOTECAS','CINEMATOGRAFICAS','JUEGOS DE AZAR'],
 'comercio':     ['COMERCIO AL POR MENOR','VENTA Y REPARACION DE VEHICULOS'],
 'equipamiento': ['SANITARIAS','EDUCACION','SERVICIOS SOCIALES','ASOCIATIVAS','OTROS SERVICIOS PERSONALES',
                  'VETERINARIAS','REPARACION DE ORDENADORES'],
}
def clas_uso(d):
    d = _na(d)
    for u, ks in USOS_KW.items():
        if any(k in d for k in ks): return u
    return None
la['uso'] = la['desc_division'].apply(clas_uso)
la = la.dropna(subset=['uso'])

gl = gpd.GeoDataFrame(la, geometry=gpd.points_from_xy(la.coordenada_x_local, la.coordenada_y_local), crs=CRS)
ju = gpd.sjoin(gl[['uso','desc_epigrafe','coordenada_x_local','coordenada_y_local','geometry']],
               gdf[['hex_id','geometry']], how='inner', predicate='within')

N_USOS, MIN = 5, 5
recs = []
for hid, sub in ju.groupby('hex_id'):
    n = len(sub)
    u = sub['uso'].value_counts()
    e = sub['desc_epigrafe'].dropna().value_counts()
    pu = (u/u.sum()).to_numpy(); S = len(e)
    Hu = -(pu*np.log(pu)).sum()
    if S > 0:
        pe = (e/e.sum()).to_numpy(); He = -(pe*np.log(pe)).sum(); simp = (pe**2).sum()
    else:
        He = simp = np.nan
    if n >= 2:
        xy = sub[['coordenada_x_local','coordenada_y_local']].to_numpy(float)
        cat = sub['uso'].to_numpy(); kk = min(6, n-1)
        _, idx = cKDTree(xy).query(xy, k=kk+1)
        m10 = float((cat[idx[:,1:]] != cat[:,None]).mean())
    else:
        m10 = np.nan
    recs.append({'hex_id':hid, 'n_loc':n,
        'm01_riqueza_usos':      u.size,
        'm02_riqueza_tipos':     S,
        'm03_shannon_usos':      Hu/np.log(N_USOS),
        'm04_shannon_tipos':     He,
        'm05_simpson_tipos':     1-simp,
        'm06_invsimpson_tipos':  1/simp if simp and simp>0 else np.nan,
        'm07_pielou_tipos':      He/np.log(S) if S>1 else 0.0,
        'm08_bergerparker_usos': 1-pu.max(),
        'm09_riqueza_ajustada':  S/np.log(n) if n>1 else 0.0,
        'm10_mezcla_espacial':   m10,
    })
divm = pd.DataFrame(recs)
mcols = [c for c in divm.columns if c.startswith('m')]
divm.loc[divm['n_loc'] < MIN, mcols] = np.nan   # pocos negocios = no fiable
print(divm[mcols].describe().round(3).T)

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import entropy

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG = BASE + r'\madrid_morfologia_h3_v10.gpkg'
LOC  = BASE + r'\Vitalidad\200085-5-censo-locales.csv'
CRS  = 'EPSG:25830'

# 1) Hexágonos y locales activos con su tipo de actividad
gdf = gpd.read_file(GPKG).to_crs(CRS)
loc = pd.read_csv(LOC, sep=';',
    usecols=['coordenada_x_local','coordenada_y_local','id_situacion_local','desc_epigrafe'],
    low_memory=False)
loc = loc[loc['id_situacion_local']==1].dropna(
        subset=['coordenada_x_local','coordenada_y_local','desc_epigrafe']).copy()

# 2) Asignar cada local a su hexágono
gl = gpd.GeoDataFrame(loc, geometry=gpd.points_from_xy(
        loc['coordenada_x_local'], loc['coordenada_y_local']), crs=CRS)
jl = gpd.sjoin(gl[['desc_epigrafe','geometry']], gdf[['hex_id','geometry']],
               how='inner', predicate='within')

# 3) VARIEDAD = diversidad (Shannon) de tipos de negocio por hexágono
def variedad(s):
    c = s.value_counts()
    return float(entropy(c)) if len(c) else np.nan     # 0 = un solo tipo; alto = muchos tipos
div = (jl.groupby('hex_id')['desc_epigrafe']
         .agg(variedad_usos=variedad, n_negocios='size').reset_index())
div.loc[div['n_negocios'] < 5, 'variedad_usos'] = np.nan   # pocos negocios = no fiable

# 4) Mapa (valor CRUDO, para que se vean las diferencias)
gm = gdf.merge(div, on='hex_id', how='left')
built = gm['cat_n_buildings'].fillna(0) >= 5
disp  = gm[built & gm['variedad_usos'].notna()]
grey  = gm[~(built & gm['variedad_usos'].notna())]

fig, ax = plt.subplots(figsize=(11,11))
vmin, vmax = np.nanpercentile(disp['variedad_usos'], (2, 98))
grey.plot(ax=ax, color='#ECECEC', linewidth=0)
disp.plot(column='variedad_usos', ax=ax, cmap='magma', vmin=vmin, vmax=vmax, linewidth=0,
          legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})

# Distritos (opcional: si osmnx falla, el mapa sale igual)
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
    dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
    for _, r in dist.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
except Exception as e:
    print('(sin distritos:', e, ')')

ax.set_title('Variety of uses · diversity of business types', fontsize=12)
ax.set_axis_off(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()
print(div['variedad_usos'].describe().round(3))

In [ ]:
import geopandas as gpd
secc = gpd.read_file(str(CENSO21_GPKG)).to_crs(CRS)
try:    cpoly = secc[secc['cod_distrito']=='01'].union_all()
except AttributeError: cpoly = secc[secc['cod_distrito']=='01'].unary_union
hc = gpd.GeoDataFrame(geometry=gdf.geometry.centroid, crs=CRS); hc['hex_id']=gdf['hex_id'].values
cids = hc[hc.within(cpoly)]['hex_id']
c = m[m['hex_id'].isin(cids)]
print('mix_usos en Centro (crudo):', round(c['mix_usos'].mean(),3),
      ' -> ~0.6 = versión vecinos | ~0.8 = entropía')
print('dim1 en Centro (percentil): ', round(c['dim1'].mean(),3))
print('dim1 mediana global:        ', round(m['dim1'].median(),3))

## 3. D2 — Amenidades cotidianas (terrazas espacio público + parques)

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG = BASE + r'\madrid_morfologia_h3_v10.gpkg'
TERR = BASE + r'\Vitalidad\200085-6-censo-locales.csv'
PARQ = BASE + r'\Vitalidad\200761-1-parques-jardines-geo.geo'
CRS  = 'EPSG:25830'

# Hexágonos
hx = gpd.read_file(GPKG).to_crs(CRS)
if 'area_km2' not in hx.columns:
    hx['area_km2'] = hx.geometry.area / 1e6

# --- Terrazas exteriores por km² ---
terr = None
for enc in ['utf-8','latin-1','cp1252']:
    try: terr = pd.read_csv(TERR, sep=';', encoding=enc, low_memory=False); break
    except UnicodeDecodeError: continue
EXTERIOR = {'Acera','Calle peatonal','Plaza peatonal','Bulevar',
            'Plaza con bandas permanentes de circulación rodada'}
te = terr[terr['desc_ubicacion_terraza'].isin(EXTERIOR)].dropna(
        subset=['coordenada_x_local','coordenada_y_local']).copy()
gt = gpd.GeoDataFrame(te, geometry=gpd.points_from_xy(
        te['coordenada_x_local'], te['coordenada_y_local']), crs=CRS)
jt = gpd.sjoin(gt[['geometry']], hx[['hex_id','geometry']], how='inner', predicate='within')
d2t = jt.groupby('hex_id').size().reset_index(name='n_terr').merge(hx[['hex_id','area_km2']], on='hex_id')
d2t['terrazas_ext_km2'] = d2t['n_terr'] / d2t['area_km2']

# --- Parques a 500 m ---
parq = gpd.read_file(PARQ)
if parq.crs is None or parq.crs.to_epsg() != 25830:
    parq = parq.to_crs(CRS)
ppts = parq.copy(); ppts['geometry'] = parq.geometry.representative_point()
hb = gpd.GeoDataFrame(hx[['hex_id']].copy(),
                      geometry=hx.geometry.centroid.buffer(500), crs=CRS)   # <-- GeoDataFrame correcto
jp = gpd.sjoin(ppts[['geometry']], hb, how='inner', predicate='within')
d2p = jp.groupby('hex_id').size().reset_index(name='n_parques_500m')

# --- Unir y mapear ---
gm = (hx.merge(d2t[['hex_id','terrazas_ext_km2']], on='hex_id', how='left')
        .merge(d2p, on='hex_id', how='left'))
gm['terrazas_ext_km2'] = gm['terrazas_ext_km2'].fillna(0)
gm['n_parques_500m']   = gm['n_parques_500m'].fillna(0)
built = gm['cat_n_buildings'].fillna(0) >= 5

dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
except Exception as e:
    print('(sin distritos:', e, ')')

def mapa_d2(col, titulo, cmap='magma'):
    disp, grey = gm[built].copy(), gm[~built].copy()
    pos = disp[col].replace(0, np.nan).dropna()
    vmax = np.nanpercentile(pos, 98) if len(pos) else 1
    fig, ax = plt.subplots(figsize=(11,11))
    grey.plot(ax=ax, color='#ECECEC', linewidth=0)
    disp.plot(column=col, ax=ax, cmap=cmap, vmin=0, vmax=vmax, linewidth=0,
              legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
    if dist is not None:
        dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
        for _, r in dist.iterrows():
            c = r.geometry.representative_point()
            ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
    ax.set_title(titulo, fontsize=12); ax.set_axis_off(); ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

mapa_d2('ext_terraces_km2', 'D2 · Exterior terraces per km²')
mapa_d2('n_parks_500m', 'D2 · Parks within 500 m')

## 4. D3 — Servicios de proximidad (mercados + parques cercanos)

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG = BASE + r'\madrid_morfologia_h3_v10.gpkg'
MERC = BASE + r'\Vitalidad\200967-4-mercados-csv.csv'
PARQ = BASE + r'\Vitalidad\200761-1-parques-jardines-geo.geo'
CRS  = 'EPSG:25830'

hx = gpd.read_file(GPKG).to_crs(CRS)
hex_pts = gpd.GeoDataFrame(hx[['hex_id']].copy(), geometry=hx.geometry.centroid, crs=CRS)

# --- Mercados: distancia al más cercano + nº a 500 m ---
merc_raw = None
for enc in ['utf-8','latin-1','cp1252']:
    try: merc_raw = pd.read_csv(MERC, sep=';', encoding=enc, low_memory=False); break
    except UnicodeDecodeError: continue
cx = [c for c in merc_raw.columns if c.upper() in ['LONGITUD','LON','X','COORDENADA_X']]
cy = [c for c in merc_raw.columns if c.upper() in ['LATITUD','LAT','Y','COORDENADA_Y']]
merc = merc_raw.dropna(subset=[cx[0],cy[0]]).copy()
merc[cx[0]] = pd.to_numeric(merc[cx[0]], errors='coerce')
merc[cy[0]] = pd.to_numeric(merc[cy[0]], errors='coerce')
merc = merc.dropna(subset=[cx[0],cy[0]])
crs_m = CRS if merc[cx[0]].median() > 1000 else 'EPSG:4326'
gdf_m = gpd.GeoDataFrame(merc, geometry=gpd.points_from_xy(merc[cx[0]], merc[cy[0]]), crs=crs_m).to_crs(CRS)

nn_m = gpd.sjoin_nearest(hex_pts, gdf_m[['geometry']], how='left',
        distance_col='d_m')[['hex_id','d_m']].groupby('hex_id').first().reset_index()
nn_m['dist_mercado_km'] = nn_m['d_m'] / 1000
hb5 = gpd.GeoDataFrame(hex_pts[['hex_id']].copy(), geometry=hex_pts.geometry.buffer(500), crs=CRS)
jm5 = gpd.sjoin(gdf_m[['geometry']], hb5, how='inner', predicate='within')
n_m5 = jm5.groupby('hex_id').size().reset_index(name='n_mercados_500m')


# --- Unir y mapear ---
gm = (hx.merge(nn_m[['hex_id','dist_mercado_km']], on='hex_id', how='left')
        .merge(n_m5, on='hex_id', how='left')
        .merge(nn_p[['hex_id','dist_parque_km']], on='hex_id', how='left'))
gm['n_mercados_500m'] = gm['n_mercados_500m'].fillna(0)
built = gm['cat_n_buildings'].fillna(0) >= 5

dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
except Exception as e:
    print('(sin distritos:', e, ')')

def mapa(col, titulo, invertir=False):
    disp = gm[built & gm[col].notna()].copy()
    grey = gm[~(built & gm[col].notna())].copy()
    vmin, vmax = np.nanpercentile(disp[col], (2, 98))
    cmap = 'magma_r' if invertir else 'magma'   # invertir -> cerca = claro
    fig, ax = plt.subplots(figsize=(11,11))
    grey.plot(ax=ax, color='#ECECEC', linewidth=0)
    disp.plot(column=col, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax, linewidth=0,
              legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
    if dist is not None:
        dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
        for _, r in dist.iterrows():
            c = r.geometry.representative_point()
            ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
    ax.set_title(titulo, fontsize=12); ax.set_axis_off(); ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

mapa('dist_mercado_km', 'D3 · Distance to market (km) · claro = closer', invertir=True)
mapa('n_mercados_500m', 'D3 · Markets within 500m')
mapa('dist_parque_km', 'D3 · Distance to park (km) · claro = closer', invertir=True)

## 5. Touristificación (Airbnb + VUT) — variable de CONTROL

Densidad de alojamiento turístico por hexágono. Es un confusor: las áreas más turísticas
tienen menos economía de barrio (los vecinos son desplazados) pero más actividad comercial bruta.

In [ ]:
ab = pd.read_csv(AIRBNB_CSV)
ab_e = ab[ab['room_type']=='Entire home/apt'].dropna(subset=['latitude','longitude']).copy()
gdf_ab = gpd.GeoDataFrame(ab_e,
    geometry=gpd.points_from_xy(ab_e['longitude'], ab_e['latitude']),
    crs='EPSG:4326').to_crs(CRS)
ja = gpd.sjoin(gdf_ab[['geometry']], gdf[['hex_id','area_km2','geometry']],
               how='inner', predicate='within')
abg = ja.groupby('hex_id').agg(n_airbnb=('hex_id','count'),
                                area_km2=('area_km2','first')).reset_index()
abg['airbnb_ent_km2'] = abg['n_airbnb']/abg['area_km2']
print(f'Airbnb entire: {len(ab_e):,}  |  hexagonos: {len(abg)}')

vut = gpd.read_file(str(VUT_SHP))
if vut.crs.to_epsg()!=25830: vut = vut.to_crs(CRS)
vut2 = vut.copy()
vut2['geometry'] = vut.geometry.apply(lambda g: Point(g.x, g.y))
vut2 = gpd.GeoDataFrame(vut2, geometry='geometry', crs=CRS)
jv = gpd.sjoin(vut2[['geometry']], gdf[['hex_id','area_km2','geometry']],
               how='inner', predicate='within')
vg = jv.groupby('hex_id').agg(n_vut=('hex_id','count'),
                               area_km2=('area_km2','first')).reset_index()
vg['vut_km2'] = vg['n_vut']/vg['area_km2']
print(f'VUT: {len(vut)}  |  hexagonos: {len(vg)}')

import matplotlib.pyplot as plt, numpy as np

# Unir las dos variables de touristificación al GeoDataFrame (sin dato = 0)
gm = (gdf.merge(abg[['hex_id','airbnb_ent_km2']], on='hex_id', how='left')
        .merge(vg[['hex_id','vut_km2']], on='hex_id', how='left'))
gm['airbnb_ent_km2'] = gm['airbnb_ent_km2'].fillna(0)
gm['vut_km2']        = gm['vut_km2'].fillna(0)
built = gm['cat_n_buildings'].fillna(0) >= 5

# Distritos (opcional: si osmnx falla, el mapa sale igual)
dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(gdf.crs)
except Exception as e:
    print('(sin distritos:', e, ')')

def mapa_tur(col, titulo, cmap='YlOrRd'):
    disp, grey = gm[built].copy(), gm[~built].copy()
    pos = disp[col].replace(0, np.nan).dropna()
    vmax = np.nanpercentile(pos, 95) if len(pos) else 1     # muy sesgado -> corto en p95
    fig, ax = plt.subplots(figsize=(11,11))
    grey.plot(ax=ax, color='#ECECEC', linewidth=0)
    disp.plot(column=col, ax=ax, cmap=cmap, vmin=0, vmax=vmax, linewidth=0,
              legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
    if dist is not None:
        dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
        for _, r in dist.iterrows():
            c = r.geometry.representative_point()
            ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
    ax.set_title(titulo, fontsize=12); ax.set_axis_off(); ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

mapa_tur('airbnb_ent_km2', 'Touristificación · Airbnb (entire) por km²')
mapa_tur('vut_km2',        'Touristificación · VUT por km²')

In [ ]:
## D3 tejido asociativo


In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG = BASE + r'\madrid_morfologia_h3_v10.gpkg'
V5   = BASE + r'\Vitalidad\madrid_vitalidad_h3_master_v5.csv'
CRS  = 'EPSG:25830'

hx = gpd.read_file(GPKG).to_crs(CRS)
m  = pd.read_csv(V5, usecols=['hex_id','asoc_km2','socios_km2'])
gm = hx.merge(m, on='hex_id', how='left')
built = gm['cat_n_buildings'].fillna(0) >= 5

dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
except Exception as e:
    print('(sin distritos:', e, ')')

def mapa(col, titulo, cmap='magma'):
    disp, grey = gm[built].copy(), gm[~built].copy()
    pos = disp[col].replace(0, np.nan).dropna()
    vmax = np.nanpercentile(pos, 95) if len(pos) else 1   # muy sesgado -> corto en p95
    fig, ax = plt.subplots(figsize=(11,11))
    grey.plot(ax=ax, color='#ECECEC', linewidth=0)
    disp.plot(column=col, ax=ax, cmap=cmap, vmin=0, vmax=vmax, linewidth=0,
              legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
    if dist is not None:
        dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
        for _, r in dist.iterrows():
            c = r.geometry.representative_point()
            ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
    ax.set_title(titulo, fontsize=12); ax.set_axis_off(); ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

mapa('asoc_km2', 'Associative fabric · Associations per km²')
mapa('socios_km2', 'Associative fabric · Members per km²')

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import rankdata

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG = BASE + r'\madrid_morfologia_h3_v10.gpkg'
V5   = BASE + r'\Vitalidad\madrid_vitalidad_h3_master_v5.csv'
CRS  = 'EPSG:25830'

def prk(s, invert=False):
    s = pd.to_numeric(s, errors='coerce'); ok = s.notna(); r = pd.Series(np.nan, index=s.index)
    if ok.sum(): r[ok] = rankdata(s[ok], method='average')/ok.sum()
    return (1-r) if invert else r

hx = gpd.read_file(GPKG).to_crs(CRS)
m  = pd.read_csv(V5, usecols=['hex_id','cv_pob'])
m['dim5'] = prk(m['cv_pob'], invert=True)     # poca variación de población = estable = más arraigo
gm = hx.merge(m[['hex_id','dim5']], on='hex_id', how='left')
built = gm['cat_n_buildings'].fillna(0) >= 5

dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
except Exception as e:
    print('(sin distritos:', e, ')')

disp, grey = gm[built & gm['dim5'].notna()], gm[~(built & gm['dim5'].notna())]
fig, ax = plt.subplots(figsize=(11,11))
vmin, vmax = np.nanpercentile(disp['dim5'], (2,98))
grey.plot(ax=ax, color='#ECECEC', linewidth=0)
disp.plot(column='dim5', ax=ax, cmap='magma', vmin=vmin, vmax=vmax, linewidth=0,
          legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
if dist is not None:
    dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
    for _, r in dist.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
ax.set_title('Residential fluidity · clear = more stable', fontsize=12)
ax.set_axis_off(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy.stats import rankdata

BASE  = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG  = BASE + r'\madrid_morfologia_h3_v10.gpkg'
CENSO = BASE + r'\Vitalidad\censo2021_seccen_madrid.gpkg'
CRS   = 'EPSG:25830'

def prk(s):
    s = pd.to_numeric(s, errors='coerce'); ok = s.notna(); r = pd.Series(np.nan, index=s.index)
    if ok.sum(): r[ok] = rankdata(s[ok], method='average')/ok.sum()
    return r

hx   = gpd.read_file(GPKG).to_crs(CRS)
secc = gpd.read_file(CENSO).to_crs(CRS)

# Sección -> hexágono (centroide de sección dentro del hex), media ponderada por población
sp = secc.copy(); sp['geometry'] = secc.geometry.centroid
jd = gpd.sjoin(sp[['pct_menores16','pct_16a64','pob_total','geometry']],
               hx[['hex_id','geometry']], how='inner', predicate='within')
def wavg(g, c):
    v = g[c].to_numpy(float); w = g['pob_total'].to_numpy(float); ok = np.isfinite(v)&np.isfinite(w)&(w>0)
    return float(np.average(v[ok], weights=w[ok])) if ok.any() else np.nan
rows = {h:(wavg(g,'pct_menores16'), wavg(g,'pct_16a64')) for h,g in jd.groupby('hex_id')}
m = pd.DataFrame([(k,a,b) for k,(a,b) in rows.items()], columns=['hex_id','pct_menores16','pct_16a64'])

# dim6 = media de los percentiles de niños y de adultos (el 65+ no suma)
m['dim6'] = pd.concat([prk(m['pct_menores16']), prk(m['pct_16a64'])], axis=1).mean(axis=1)
gm = hx.merge(m[['hex_id','dim6']], on='hex_id', how='left')
built = gm['cat_n_buildings'].fillna(0) >= 5

dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
except Exception as e:
    print('(sin distritos:', e, ')')

disp, grey = gm[built & gm['dim6'].notna()], gm[~(built & gm['dim6'].notna())]
fig, ax = plt.subplots(figsize=(11,11))
vmin, vmax = np.nanpercentile(disp['dim6'], (2,98))
grey.plot(ax=ax, color='#ECECEC', linewidth=0)
disp.plot(column='dim6', ax=ax, cmap='magma', vmin=vmin, vmax=vmax, linewidth=0,
          legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
if dist is not None:
    dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
    for _, r in dist.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
ax.set_title('Dim 6 · Age (children + adults 16-64)', fontsize=12)
ax.set_axis_off(); ax.set_aspect('equal'); plt.tight_layout(); plt.show()

## 6. Controles socioeconómicos (renta, densidad, centralidad)

In [ ]:
# Renta ADRH 2022
adrh = pd.read_csv(ADRH_CSV, sep=';', low_memory=False)
mr = adrh[(adrh['Municipios'].astype(str).str.startswith('28079')) &
          (adrh['Periodo']==2022) &
          (adrh['Indicadores de renta media y mediana']=='Renta neta media por persona')
         ].dropna(subset=['Secciones']).copy()
mr['CUSEC'] = mr['Secciones'].astype(str).str[:10].str.zfill(10)
mr['renta_persona'] = pd.to_numeric(mr['Total'], errors='coerce')*1000
mr = mr[['CUSEC','renta_persona']].dropna()

# Censo 2021 (densidad) + mapa sección→hex
secc = gpd.read_file(str(CENSO21_GPKG)).to_crs(CRS)
secc['densidad_pob_km2'] = secc['pob_total'] / (secc.geometry.area/1e6)
secc_pts = secc.copy(); secc_pts['geometry'] = secc.geometry.centroid
js = gpd.sjoin(secc_pts[['CUSEC','pob_total','densidad_pob_km2','geometry']],
               gdf[['hex_id','geometry']], how='inner', predicate='within')
js['CUSEC'] = js['CUSEC'].astype(str).str.zfill(10)

def wavg(g,var,w='pob_total'):
    v=g[[var,w]].dropna()
    if v.empty or v[w].sum()==0: return np.nan
    return float(np.average(v[var].values, weights=v[w].values))
dens = js.groupby('hex_id').apply(lambda g: wavg(g,'densidad_pob_km2')).reset_index(name='densidad_pob_km2')
rh = js[['CUSEC','hex_id','pob_total']].merge(mr, on='CUSEC', how='inner')
renta = (rh.groupby('hex_id')
         .apply(lambda g: np.average(g['renta_persona'],weights=g['pob_total'])
                if g['pob_total'].sum()>0 else np.nan)
         .reset_index(name='renta_persona'))

# Centralidad
cent = pd.read_csv(CENTRAL_CSV)
print(f'Renta: {len(renta)}  Densidad: {len(dens)}  Centralidad: {len(cent)}')

In [ ]:
import geopandas as gpd, pandas as pd, numpy as np, matplotlib.pyplot as plt

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'
GPKG = BASE + r'\madrid_morfologia_h3_v10.gpkg'
OCIO = BASE + r'\Vitalidad\vitalidad_ocio_hex.csv'
CRS  = 'EPSG:25830'

hx = gpd.read_file(GPKG).to_crs(CRS)
oc = pd.read_csv(OCIO)   # hex_id, nocturno_km2, cultura_km2, restauracion_km2
gm = hx.merge(oc, on='hex_id', how='left')
for c in ['nocturno_km2','cultura_km2','restauracion_km2']:
    gm[c] = gm[c].fillna(0)
built = gm['cat_n_buildings'].fillna(0) >= 5

dist = None
try:
    import osmnx as ox
    ox.settings.use_cache = True
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level':'9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(CRS)
except Exception as e:
    print('(sin distritos:', e, ')')

def mapa(col, titulo, cmap='magma'):
    disp, grey = gm[built].copy(), gm[~built].copy()
    pos = disp[col].replace(0, np.nan).dropna()
    vmax = np.nanpercentile(pos, 95) if len(pos) else 1     # muy sesgado -> corto en p95
    fig, ax = plt.subplots(figsize=(11,11))
    grey.plot(ax=ax, color='#ECECEC', linewidth=0)
    disp.plot(column=col, ax=ax, cmap=cmap, vmin=0, vmax=vmax, linewidth=0,
              legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30})
    if dist is not None:
        dist.boundary.plot(ax=ax, color='black', linewidth=0.8, alpha=0.6)
        for _, r in dist.iterrows():
            c = r.geometry.representative_point()
            ax.annotate(r['name'], (c.x, c.y), fontsize=6, ha='center', va='center', fontweight='bold',
                        bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
    ax.set_title(titulo, fontsize=12); ax.set_axis_off(); ax.set_aspect('equal')
    plt.tight_layout(); plt.show()

mapa('nocturno_km2',     'Ocio nocturno por km²  (dim8)')
mapa('cultura_km2',      'Cultura por km²  (era dim9)')
mapa('restauracion_km2', 'Restauración por km²  (era dim10)')

## 7. Ensamblaje del dataset maestro v4

In [ ]:
m = gdf[['hex_id','tipologia_label','morfologia_continua','area_km2','dist_centro_km']].copy()
m = (m
  .merge(d1[['hex_id','barrio_km2','pct_barrio','pct_host','div_actividad','n_jacobs','n_barrio','n_host']], on='hex_id', how='left')
  .merge(d2t[['hex_id','terrazas_ext_km2']], on='hex_id', how='left')
  .merge(d2p, on='hex_id', how='left')
  .merge(nn_m[['hex_id','dist_mercado_km']], on='hex_id', how='left')
  .merge(n_m5, on='hex_id', how='left')
  .merge(nn_p[['hex_id','dist_parque_km']], on='hex_id', how='left')
  .merge(abg[['hex_id','airbnb_ent_km2','n_airbnb']], on='hex_id', how='left')
  .merge(vg[['hex_id','vut_km2','n_vut']], on='hex_id', how='left')
  .merge(dens, on='hex_id', how='left')
  .merge(renta, on='hex_id', how='left')
  .merge(cent[['hex_id','log_bc_mean']], on='hex_id', how='left'))

# Conteos/densidades vacíos = 0
for c in ['barrio_km2','pct_barrio','pct_host','div_actividad','n_jacobs','n_barrio','n_host',
          'terrazas_ext_km2','n_parques_500m','n_mercados_500m',
          'airbnb_ent_km2','n_airbnb','vut_km2','n_vut']:
    if c in m.columns: m[c] = m[c].fillna(0)
m['log_bc_mean'] = m['log_bc_mean'].fillna(0)

# Transformaciones para controles
m['log_renta']    = np.log(m['renta_persona'].clip(lower=1))
m['log_densidad'] = np.log1p(m['densidad_pob_km2'])
m['log_airbnb']   = np.log1p(m['airbnb_ent_km2'])
m['log_vut']      = np.log1p(m['vut_km2'])
m['espontaneo']   = m['tipologia_label'].map({'Espontáneo':1,'Planificado':0})

print(f'Master v4: {len(m)} hexagonos x {len(m.columns)} variables')
print(m[['barrio_km2','pct_barrio','div_actividad','terrazas_ext_km2',
         'airbnb_ent_km2','vut_km2','renta_persona','log_bc_mean']].describe().round(2))

In [ ]:
## 7.b Exploración rápida — ceros y NaN por columna

num = m.select_dtypes('number')
rep = pd.DataFrame({
    'pct_ceros': (num == 0).mean().mul(100).round(1),
    'pct_nan':   num.isna().mean().mul(100).round(1),
    'min':       num.min().round(2),
    'media':     num.mean().round(2),
    'max':       num.max().round(2),
})
rep = rep.sort_values('pct_ceros', ascending=False)
print(f'Dataset: {m.shape[0]} hexagonos x {m.shape[1]} columnas\n')
print(rep.to_string())

In [ ]:
import pandas as pd, geopandas as gpd, numpy as np
base = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA'

loc = pd.read_csv(base + r'\Vitalidad\200085-5-censo-locales.csv', sep=';', encoding='latin-1',
                  usecols=['desc_situacion_local','desc_epigrafe','coordenada_x_local','coordenada_y_local'])
loc = loc[loc['desc_situacion_local'].astype(str).str.contains('Abierto', na=False)].copy()
loc['x'] = pd.to_numeric(loc['coordenada_x_local'], errors='coerce')
loc['y'] = pd.to_numeric(loc['coordenada_y_local'], errors='coerce')
loc = loc.dropna(subset=['x','y'])

cats = {
 'nocturno': ['DISCOTECAS Y SALAS DE BAILE','BAR ESPECIAL SIN ACTUACIONES','BAR ESPECIAL CON ACTUACIONES',
              'CAFE ESPECTACULO','SALAS DE FIESTA SIN RESTAURACION','SALAS DE FIESTA CON RESTAURACION','TABERNA'],
 'cultura':  ['TEATRO Y ACTIVIDADES ESCENICAS REALIZADAS EN DIRECTO','ACTIVIDADES DE CREACION, ARTISTICAS Y ESPECTACULOS',
              'SALA DE EXPOSICIONES Y GALERIAS DE ARTE CON VENTA','ACTIVIDADES DE GRABACION DE SONIDO Y EDICION MUSICAL',
              'ACTIVIDADES DE BIBLIOTECAS, ARCHIVOS, MUSEOS Y DE GALERIAS Y SALAS DE EXPOSICIONES SIN VENTA'],
 'restauracion': ['BAR RESTAURANTE','BAR CON COCINA','CAFETERIA','RESTAURANTE','BAR SIN COCINA','RESTAURANTES DE COMIDA RAPIDA'],
}

gl   = gpd.GeoDataFrame(loc, geometry=gpd.points_from_xy(loc['x'], loc['y']), crs=25830)
hexg = gpd.read_file(base + r'\madrid_morfologia_h3_v10.gpkg')[['hex_id','geometry']].to_crs(25830)
hexg['area_km2'] = hexg.geometry.area / 1e6
j = gpd.sjoin(gl, hexg, how='inner', predicate='within')

for cat, eps in cats.items():
    cnt = j[j['desc_epigrafe'].isin(eps)].groupby('hex_id').size()
    hexg[cat + '_km2'] = (hexg['hex_id'].map(cnt).fillna(0) / hexg['area_km2'])

hexg[['hex_id','nocturno_km2','cultura_km2','restauracion_km2']].to_csv(
    base + r'\Vitalidad\vitalidad_ocio_hex.csv', index=False)
print('guardado vitalidad_ocio_hex.csv'); print(hexg[['nocturno_km2','cultura_km2','restauracion_km2']].describe().round(1))

## 8. Construcción del índice de vitalidad VECINAL v4 — versión limpia

Percentile rank [0,1]; (↓) = invertido; dimensión = media de sus variables; índice = media de dimensiones.

**Criterio de esta versión:** Airbnb y VUT se conservan solo como variables de control/validación, pero no forman parte del índice. También se elimina `n_mercados_500m` de la métrica por su escasez extrema; se mantiene `dist_mercado_km` como medida continua de proximidad.


In [ ]:
import pandas as pd, numpy as np, geopandas as gpd
from scipy.stats import rankdata

base = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\Vitalidad'
m = pd.read_csv(base + r'\madrid_vitalidad_h3_master_v5.csv')
m = m.merge(pd.read_csv(base + r'\madrid_vitalidad_h3_master.csv')[['hex_id','pct_65mas']], on='hex_id', how='left')
m = m.merge(pd.read_csv(base + r'\vitalidad_ocio_hex.csv'), on='hex_id', how='left')

# >>> D1: mezcla de USOS (calculada en celdas 5-8) <<<
m = m.merge(d1[['hex_id','mix_usos','n_usos']], on='hex_id', how='left')
m['mix_usos'] = m['mix_usos'].fillna(0)

# >>> D6: EDAD VITAL — premiar niños (<16) y adultos (16-64); el 65+ no suma <<<
secc = gpd.read_file(str(CENSO21_GPKG)).to_crs(CRS)
secc_pts = secc.copy(); secc_pts['geometry'] = secc.geometry.centroid
jd = gpd.sjoin(secc_pts[['pct_menores16','pct_16a64','pob_total','geometry']],
               gdf[['hex_id','geometry']], how='inner', predicate='within')
def wavg(grp, col):
    v = grp[col].to_numpy(float); w = grp['pob_total'].to_numpy(float)
    ok = np.isfinite(v) & np.isfinite(w) & (w > 0)
    return float(np.average(v[ok], weights=w[ok])) if ok.any() else np.nan
rows = {hid: (wavg(grp,'pct_menores16'), wavg(grp,'pct_16a64'))
        for hid, grp in jd.groupby('hex_id')}
hex_edad = pd.DataFrame([(k,a,b) for k,(a,b) in rows.items()],
                        columns=['hex_id','pct_menores16','pct_16a64'])
m = m.merge(hex_edad, on='hex_id', how='left')

if 'log_airbnb' not in m.columns and 'airbnb_ent_km2' in m.columns:
    m['log_airbnb'] = np.log1p(m['airbnb_ent_km2'])

def prk(s, invert=False):
    s = pd.to_numeric(s, errors='coerce'); valid = s.notna()
    r = pd.Series(np.nan, index=s.index)
    if valid.sum() > 0: r[valid] = rankdata(s[valid], method='average') / valid.sum()
    return (1 - r) if invert else r
def dim(*cols): return pd.concat(cols, axis=1).mean(axis=1)

m['dim1'] = prk(m['mix_usos'])                                          # variedad de usos
m['dim2'] = dim(prk(m['terrazas_ext_km2']), prk(m['n_parques_500m']))   # amenidades
m['dim4'] = prk(m['asoc_km2'])                                          # asociaciones
m['dim5'] = prk(m['cv_pob'])                                            # rotación / dinamismo
m['dim6'] = dim(prk(m['pct_menores16']), prk(m['pct_16a64']))           # juventud
m['dim8'] = dim(prk(m['nocturno_km2']), prk(m['restauracion_km2']))     # ocio + restauración  ← cambiada
dims = ['dim1','dim2','dim4','dim5','dim6','dim8']

m['vitalidad'] = m[dims].mean(axis=1)
m['vitalidad_vecinal'] = m['vitalidad']

print('Índice con D1 = mezcla de usos y D6 = edad (niños + adultos 16-64):')
print(m[['mix_usos','pct_menores16','pct_16a64'] + dims + ['vitalidad']].describe().round(3))

### Celda de índice duplicada desactivada

En el notebook anterior esta celda volvía a recalcular `m['vitalidad']` e introducía de nuevo Airbnb/VUT mediante `dim7`. Queda desactivada para evitar que pise el índice limpio definido arriba.


### Celda de índice duplicada desactivada

En el notebook anterior esta celda volvía a recalcular `m['vitalidad']` e introducía de nuevo Airbnb/VUT mediante `dim7`. Queda desactivada para evitar que pise el índice limpio definido arriba.


In [ ]:
import geopandas as gpd, numpy as np, matplotlib.pyplot as plt
import osmnx as ox
from pathlib import Path

g = gpd.read_file(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\madrid_morfologia_h3_v10.gpkg').to_crs(25830)
g = g.merge(m[['hex_id','vitalidad']], on='hex_id', how='left')

built = g['cat_n_buildings'].fillna(0) >= 5
disp  = g[built].copy(); grey = g[~built].copy()

fig, ax = plt.subplots(figsize=(12, 12))
vmin, vmax = np.nanpercentile(disp['vitalidad'].astype(float).dropna(), (2, 98))
grey.plot(ax=ax, color='#E8E8E8', linewidth=0)
disp.plot(column='vitalidad', ax=ax, cmap='magma', vmin=vmin, vmax=vmax, linewidth=0,
          legend=True, legend_kwds={'shrink':0.6,'orientation':'horizontal','pad':0.01,'aspect':30},
          missing_kwds={'color':'#E8E8E8'})

# --- distritos ---
ox.settings.use_cache = True
try:
    dist = ox.features_from_place('Madrid, Spain', tags={'admin_level': '9'})
    dist = dist[dist.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']].to_crs(25830)
    dist.boundary.plot(ax=ax, color='black', linewidth=0.9, alpha=0.65)
    for _, r in dist.iterrows():                 # nombres de distrito (opcional)
        c = r.geometry.centroid
        ax.annotate(r['name'], (c.x, c.y), fontsize=6.5, ha='center', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.1', fc='white', alpha=0.5, lw=0))
except Exception as e:
    print('sin distritos (¿internet/osmnx?):', e)

ax.set_title('Vitalidad por hexágono · índice limpio sin Airbnb/VUT · gris = no urbano excluido', fontsize=12)
ax.set_axis_off(); ax.set_aspect('equal')
plt.tight_layout(); Path('imagenes').mkdir(exist_ok=True)
plt.savefig('imagenes/mapa_vitalidad_distritos_limpio.png', dpi=170, bbox_inches='tight'); plt.show()

In [ ]:
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import math
import osmnx as ox
from pathlib import Path

# -----------------------------
# 1) Cargar geometría hexagonal
# -----------------------------
g = gpd.read_file(
    r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\madrid_morfologia_h3_v10.gpkg'
).to_crs(25830)

# -----------------------------
# 2) Dimensiones activas actuales
# -----------------------------
dims = ['dim1', 'dim2', 'dim4', 'dim5', 'dim6', 'dim8']

titulos = {
    'dim1': 'Dim 1 · Mezcla de usos',
    'dim2': 'Dim 2 · Terrazas + parques',
    'dim4': 'Dim 4 · Tejido asociativo',
    'dim5': 'Dim 5 · Estabilidad residencial',
    'dim6': 'Dim 6 · Edad (niños + adultos)',
    'dim8': 'Dim 8 · Ocio nocturno',
    'vitalidad': 'Índice final de vitalidad'
}

# -----------------------------
# 3) Unir índice al GeoDataFrame
# -----------------------------
cols_merge = ['hex_id'] + dims + ['vitalidad']
gm = g.merge(m[cols_merge], on='hex_id', how='left')

# Separar hexágonos urbanos / no urbanos
built = gm['cat_n_buildings'].fillna(0) >= 5
disp = gm[built].copy()
grey = gm[~built].copy()

# -----------------------------
# 4) Cargar distritos desde OSMnx
# -----------------------------
ox.settings.use_cache = True

try:
    dist = ox.features_from_place(
        'Madrid, Spain',
        tags={'admin_level': '9'}
    )
    dist = dist[
        dist.geometry.type.isin(['Polygon', 'MultiPolygon'])
    ][['name', 'geometry']].to_crs(25830)

    print('Distritos cargados:', len(dist))

except Exception as e:
    dist = None
    print('sin distritos (¿internet/osmnx?):', e)

# -----------------------------
# 5) Función para dibujar distritos
# -----------------------------
def dibujar_distritos(ax):
    if dist is None:
        return
    
    dist.boundary.plot(
        ax=ax,
        color='black',
        linewidth=0.9,
        alpha=0.65
    )
    
    for _, r in dist.iterrows():
        c = r.geometry.representative_point()
        ax.annotate(
            r['name'],
            (c.x, c.y),
            fontsize=6.5,
            ha='center',
            va='center',
            fontweight='bold',
            bbox=dict(
                boxstyle='round,pad=0.1',
                fc='white',
                alpha=0.5,
                lw=0
            )
        )

# -----------------------------
# 6) Carpeta de salida
# -----------------------------
out_dir = Path(
    r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\Vitalidad\imagenes'
)
out_dir.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 7) Mapas individuales por dimensión
# -----------------------------
for d in dims:
    fig, ax = plt.subplots(figsize=(12, 12))
    
    vals = disp[d].astype(float).dropna()
    vmin, vmax = np.nanpercentile(vals, (2, 98))
    
    grey.plot(
        ax=ax,
        color='#E8E8E8',
        linewidth=0
    )
    
    disp.plot(
        column=d,
        ax=ax,
        cmap='magma',
        vmin=vmin,
        vmax=vmax,
        linewidth=0,
        legend=True,
        legend_kwds={
            'shrink': 0.6,
            'orientation': 'horizontal',
            'pad': 0.01,
            'aspect': 30
        },
        missing_kwds={'color': '#E8E8E8'}
    )
    
    dibujar_distritos(ax)
    
    ax.set_title(titulos.get(d, d), fontsize=12)
    ax.set_axis_off()
    ax.set_aspect('equal')
    
    plt.tight_layout()
    plt.savefig(out_dir / f'mapa_{d}_distritos.png', dpi=170, bbox_inches='tight')
    plt.show()

# -----------------------------
# 8) Mapa del índice final
# -----------------------------
fig, ax = plt.subplots(figsize=(12, 12))

vals = disp['vitalidad'].astype(float).dropna()
vmin, vmax = np.nanpercentile(vals, (2, 98))

grey.plot(
    ax=ax,
    color='#E8E8E8',
    linewidth=0
)

disp.plot(
    column='vitalidad',
    ax=ax,
    cmap='magma',
    vmin=vmin,
    vmax=vmax,
    linewidth=0,
    legend=True,
    legend_kwds={
        'shrink': 0.6,
        'orientation': 'horizontal',
        'pad': 0.01,
        'aspect': 30
    },
    missing_kwds={'color': '#E8E8E8'}
)

dibujar_distritos(ax)

ax.set_title(
    'Vitalidad por hexágono · índice actual · gris = no urbano excluido',
    fontsize=12
)
ax.set_axis_off()
ax.set_aspect('equal')

plt.tight_layout()
plt.savefig(out_dir / 'mapa_vitalidad_final_distritos.png', dpi=170, bbox_inches='tight')
plt.show()

# -----------------------------
# 9) Panel resumen de dimensiones
# -----------------------------
n = len(dims)
ncols = 3
nrows = math.ceil(n / ncols)

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 6*nrows))
axes = axes.flatten()

for i, d in enumerate(dims):
    vals = disp[d].astype(float).dropna()
    vmin, vmax = np.nanpercentile(vals, (2, 98))
    
    grey.plot(
        ax=axes[i],
        color='#E8E8E8',
        linewidth=0
    )
    
    disp.plot(
        column=d,
        ax=axes[i],
        cmap='magma',
        vmin=vmin,
        vmax=vmax,
        linewidth=0,
        legend=True,
        legend_kwds={
            'shrink': 0.55,
            'orientation': 'horizontal',
            'pad': 0.01,
            'aspect': 25
        },
        missing_kwds={'color': '#E8E8E8'}
    )
    
    dibujar_distritos(axes[i])
    
    axes[i].set_title(titulos.get(d, d), fontsize=10)
    axes[i].set_axis_off()
    axes[i].set_aspect('equal')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.savefig(out_dir / 'panel_dimensiones_distritos.png', dpi=170, bbox_inches='tight')
plt.show()

In [ ]:
from pathlib import Path
import json
import re

base = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\Vitalidad')

salida = base / 'BUSQUEDA_div_actividad_COMPLETA.txt'

patrones = [
    'div_actividad',
    'n_div',
    'd_comercio',
    'actividad',
    'categoria',
    'category',
    'tipo',
    'loc_km2',
    'pct_pc',
    'n_jacobs'
]

with open(salida, 'w', encoding='utf-8') as out:
    for f in list(base.glob('*.ipynb')) + list(base.glob('*.py')):
        try:
            if f.suffix == '.ipynb':
                nb = json.loads(f.read_text(encoding='utf-8', errors='ignore'))
                cells = nb.get('cells', [])
                
                for idx, cell in enumerate(cells):
                    src = ''.join(cell.get('source', []))
                    if any(p in src for p in patrones):
                        out.write('\n' + '='*120 + '\n')
                        out.write(f'ARCHIVO: {f.name}\n')
                        out.write(f'CELDA: {idx}\n')
                        out.write(f'TIPO: {cell.get("cell_type")}\n')
                        out.write('-'*120 + '\n')
                        out.write(src)
                        out.write('\n')
            
            else:
                txt = f.read_text(encoding='utf-8', errors='ignore')
                if any(p in txt for p in patrones):
                    out.write('\n' + '='*120 + '\n')
                    out.write(f'ARCHIVO: {f.name}\n')
                    out.write('-'*120 + '\n')
                    out.write(txt)
                    out.write('\n')
                    
        except Exception as e:
            out.write(f'\nERROR en {f.name}: {e}\n')

print('Archivo creado:')
print(salida)

## 9. VALIDACIÓN — correlación con touristificación

Airbnb/VUT ya no entran en el índice. Esta sección comprueba si el nuevo índice sigue correlacionando indirectamente con touristificación por otras vías, como restauración, cultura, ocio o centralidad.


In [ ]:
print('=== CORRELACIÓN CON TOURISTIFICACIÓN (Airbnb) ===')
print()
sub = m[m['log_airbnb']>0].copy()
for var, lab in [('vitalidad_vecinal','Vitalidad vecinal v4'),
                 ('dim1','  D1 Activación vecinal'),
                 ('pct_barrio','  pct_barrio (share proximidad)'),
                 ('barrio_km2','  barrio_km2')]:
    r,p = stats.pearsonr(sub['log_airbnb'], sub[var])
    sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
    print(f'  log_airbnb vs {lab:<32}: r={r:+.3f} {sig}')

# Comparar con v2 si existe
v2p = VITALIDAD / 'madrid_vitalidad_h3_master_v2.csv'
if v2p.exists():
    v2 = pd.read_csv(v2p)[['hex_id','indice_vitalidad']].rename(columns={'indice_vitalidad':'iv_v2'})
    cmp = m[['hex_id','vitalidad_vecinal','log_airbnb']].merge(v2, on='hex_id')
    cmp_s = cmp[cmp['log_airbnb']>0]
    r2,_ = stats.pearsonr(cmp_s['log_airbnb'], cmp_s['iv_v2'])
    rv,_ = stats.pearsonr(cmp_s['log_airbnb'], cmp_s['vitalidad_vecinal'])
    print()
    print(f'  >> v2 (animación comercial)  vs Airbnb: r={r2:+.3f}')
    print(f'  >> v4 (vitalidad vecinal)    vs Airbnb: r={rv:+.3f}')
    rr,_ = stats.pearsonr(cmp['iv_v2'], cmp['vitalidad_vecinal'])
    print(f'  >> correlación v2 vs v4: r={rr:+.3f}')

print()
print('=== TOP 10 hexagonos vitalidad vecinal v4 ===')
top_cols = [c for c in ['hex_id','tipologia_label','vitalidad_vecinal','barrio_km2','pct_barrio','airbnb_ent_km2','dist_centro_km'] if c in m.columns]
top = m.nlargest(10,'vitalidad_vecinal')[top_cols]
print(top.round(2).to_string())

In [ ]:
# Medias por cuartil de Airbnb: v2 vs v4
fig, axes = plt.subplots(1,3, figsize=(18,5))

if v2p.exists():
    cmp['q_ab'] = pd.qcut(cmp['log_airbnb'], q=5, duplicates='drop')
    g = cmp.groupby('q_ab', observed=True)[['iv_v2','vitalidad_vecinal']].mean()
    x = range(len(g))
    axes[0].plot(x, g['iv_v2'], 'o-', color='#d7191c', label='v2 (animación comercial)', lw=2)
    axes[0].plot(x, g['vitalidad_vecinal'], 's-', color='#2b83ba', label='v4 (vitalidad vecinal)', lw=2)
    axes[0].set_xlabel('Quintil de densidad Airbnb →')
    axes[0].set_ylabel('Índice medio')
    axes[0].set_title('v2 sube con turismo; v4 no\n(validación del rediseño)')
    axes[0].legend(fontsize=9)
    axes[0].set_xticks(list(x))

# Scatter Airbnb vs v4
s2 = m[m['log_airbnb']>0]
col = m['tipologia_label'].map({'Espontáneo':'#d7191c','Planificado':'#2b83ba','Transición':'#aaa'})
axes[1].scatter(s2['log_airbnb'], s2['vitalidad_vecinal'],
                c=col.loc[s2.index], alpha=0.25, s=6)
z = np.polyfit(s2['log_airbnb'], s2['vitalidad_vecinal'],1)
xl = np.linspace(s2['log_airbnb'].min(), s2['log_airbnb'].max(),50)
axes[1].plot(xl, np.polyval(z,xl), 'black', lw=2)
axes[1].set_xlabel('log(Airbnb entire/km²)'); axes[1].set_ylabel('Vitalidad vecinal v4')
axes[1].set_title('Airbnb vs v4')

# Distribución por tipología
for tip,c in [('Espontáneo','#d7191c'),('Planificado','#2b83ba')]:
    axes[2].hist(m[m['tipologia_label']==tip]['vitalidad_vecinal'].dropna(),
                 bins=30, alpha=0.55, color=c, label=tip, density=True)
axes[2].set_xlabel('Vitalidad vecinal v4'); axes[2].legend()
axes[2].set_title('Distribución por tipología')
plt.tight_layout()
plt.savefig('imagenes_v4/validacion_v4.png', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cambiar a la clasificación NUEVA (Random Forest v5) ───────────────────────
import geopandas as gpd
RF = gpd.read_file(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\madrid_clasificacion_final_v5.gpkg')[['hex_id','tipologia_rf']]
RF['hex_id'] = RF['hex_id'].astype(str)
mapa_rf = dict(zip(RF['hex_id'], RF['tipologia_rf']))

# Sobrescribir en m (lo usan descriptivos y PSM)
m['hex_id'] = m['hex_id'].astype(str)
m['tipologia_label'] = m['hex_id'].map(mapa_rf)
m['espontaneo']      = m['tipologia_label'].map({'Espontáneo':1, 'Planificado':0})   # Transición -> NaN

# Sobrescribir también en gdf (lo usa el Método B / frontera)
gdf['hex_id'] = gdf['hex_id'].astype(str)
gdf['tipologia_label'] = gdf['hex_id'].map(mapa_rf)

print('Clasificación cambiada a RF v5:')
print('  m :', m['tipologia_label'].value_counts(dropna=False).to_dict())

## 10. Estadística descriptiva por tipología

In [ ]:
vd = {'vitalidad_vecinal':'Vitalidad v4',
      'dim1':'D1 Variedad de usos',
      'dim2':'D2 Amenidades',
      'dim4':'D4 Tejido asociativo',
      'dim5':'D5 Rotación / dinamismo',        
      'dim6':'D6 Juventud (niños+adultos)',
      'dim8':'D8 Ocio + restauración',
}
rows=[]
for v,l in vd.items():
    if v not in m.columns: continue
    r={'Variable':l}
    for tip in ['Espontáneo','Planificado','Transición']:
        s=m[m['tipologia_label']==tip][v].dropna()
        r[tip]=f'{s.mean():.3f}' if abs(s.mean())<100 else f'{s.mean():.0f}'
    a=m[m['tipologia_label']=='Espontáneo'][v].dropna()
    b=m[m['tipologia_label']=='Planificado'][v].dropna()
    if len(a)>1 and len(b)>1:
        _,p=stats.ttest_ind(a,b)
        d=(a.mean()-b.mean())/np.sqrt((a.std()**2+b.std()**2)/2)
        r['Cohen d']=f'{d:.3f}'
        r['p']=f'{p:.4f}'+('***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else '')
    rows.append(r)
print(pd.DataFrame(rows).set_index('Variable').to_string())

## 11. Mapas

In [ ]:
gm = gdf.merge(m[['hex_id','vitalidad_vecinal','dim1','dim2','dim6','airbnb_ent_km2']], on='hex_id', how='left')
fig, axes = plt.subplots(2,3, figsize=(18,11)); axes=axes.flatten()
gm.plot(column='vitalidad_vecinal', ax=axes[0], cmap='RdYlGn', vmin=0.2, vmax=0.8, legend=True,
        legend_kwds={'shrink':0.7}, linewidth=0.05, edgecolor='none', missing_kwds={'color':'lightgrey'})
axes[0].set_title('Vitalidad vecinal v4 limpia'); axes[0].set_axis_off()
for i,(c,l) in enumerate(zip(['dim1','dim2','dim6'],
        ['D1 Mezcla de usos','D2 Amenidades','D6 Edad (niños+adultos)'])):
    gm.plot(column=c, ax=axes[i+1], cmap='RdYlGn', vmin=0.2, vmax=0.8, legend=True,
            legend_kwds={'shrink':0.7}, linewidth=0.05, edgecolor='none', missing_kwds={'color':'lightgrey'})
    axes[i+1].set_title(l); axes[i+1].set_axis_off()
gm.plot(column='airbnb_ent_km2', ax=axes[4], cmap='YlOrRd', legend=True,
        legend_kwds={'shrink':0.7}, linewidth=0.05, edgecolor='none',
        vmax=gm['airbnb_ent_km2'].quantile(0.95))
axes[4].set_title('Airbnb (touristificación)'); axes[4].set_axis_off()
cmap={'Espontáneo':'#d7191c','Planificado':'#2b83ba','Transición':'#ddd'}
gm['_c']=gm['tipologia_label'].map(cmap).fillna('#ccc')
gm.plot(color=gm['_c'], ax=axes[5], linewidth=0.05, edgecolor='none')
axes[5].legend(handles=[mpatches.Patch(color=v,label=k) for k,v in cmap.items()], fontsize=8, loc='lower right')
axes[5].set_title('Tipología morfológica'); axes[5].set_axis_off()
plt.tight_layout(); plt.savefig('imagenes_v4/mapas_v4_limpio.png', dpi=180, bbox_inches='tight'); plt.show()

## 12. MÉTODO A — Matching (PSM) con control de touristificación

Propensity score: `Logit(espontáneo | log_renta, log_densidad, dist_centro_km, log_bc_mean, log_airbnb)`.
La inclusión de `log_airbnb` evita comparar un espontáneo touristificado con un planificado virgen.
Matching greedy 1:1, caliper = 0.2·SD(PS). Outcome: **vitalidad vecinal v4**.

In [ ]:
OUT  = 'vitalidad_vecinal'
CTRL = ['log_renta','log_densidad','dist_centro_km','log_bc_mean']   # renta + densidad + centralidad (sin turismo)
K    = 3                                                             # nº de vecinos (pon 1 para el original)

df = m[m['tipologia_label'].isin(['Espontáneo','Planificado'])].dropna(subset=[OUT,'espontaneo']+CTRL).copy()
print(f'N: {len(df)}  Esp={int(df.espontaneo.sum())}  Plan={int((df.espontaneo==0).sum())}')

sc = StandardScaler(); Xs = sc.fit_transform(df[CTRL].values); T = df['espontaneo'].values
lp = LogisticRegression(max_iter=1000, random_state=42).fit(Xs, T)
df['ps'] = lp.predict_proba(Xs)[:,1]

CAL = 0.2*df['ps'].std()
it = df[df.espontaneo==1].index.values; ic = df[df.espontaneo==0].index.values
pt = df.loc[it,'ps'].values; pc = df.loc[ic,'ps'].values
tree = cKDTree(pc.reshape(-1,1))
dist, nn = tree.query(pt.reshape(-1,1), k=K)           # K vecinos más cercanos, CON reemplazo
if K == 1: dist, nn = dist.reshape(-1,1), nn.reshape(-1,1)

diffs, itm, icm = [], [], []
for i in range(len(it)):
    sel = nn[i][dist[i] <= CAL]                        # solo vecinos dentro del caliper
    if len(sel) == 0: continue
    cids = ic[sel]
    diffs.append(df.loc[it[i], OUT] - df.loc[cids, OUT].mean())   # espontáneo - media de sus K controles
    itm.append(it[i]); icm.extend(cids.tolist())
diffs = np.array(diffs)
att = diffs.mean(); se = diffs.std(ddof=1)/np.sqrt(len(diffs))
tstat = att/se; patt = 2*(1 - stats.t.cdf(abs(tstat), df=len(diffs)-1))
ci = (att - 1.96*se, att + 1.96*se)
sig = lambda p:'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
print(f'\nPSM k={K} (con reemplazo, outcome={OUT}):')
print(f'  Espontáneos emparejados: {len(diffs)} de {len(it)}')
print(f'  ATT = {att:+.4f}  {sig(patt)}  p={patt:.4f}  IC95=[{ci[0]:.4f},{ci[1]:.4f}]')
print('  OJO: con reemplazo la SE naive subestima (controles reutilizados) -> p optimista. Para inferencia seria, bootstrap.')

# Balance
def smd(a,b):
    s=np.sqrt((a.std()**2+b.std()**2)/2); return abs(a.mean()-b.mean())/s if s>0 else 0
print('\n  Balance SMD (pre → post):')
for v in CTRL+['ps']:
    pre  = smd(df[df.espontaneo==1][v], df[df.espontaneo==0][v])
    post = smd(df.loc[itm,v], df.loc[icm,v])
    print(f'    {v:<16} {pre:.3f} → {post:.3f}')
att_psm, p_psm, ci_psm = att, patt, ci

In [ ]:
OUT  = 'vitalidad_vecinal'
CTRL = ['log_renta','log_densidad','dist_centro_km','log_bc_mean']   # renta + densidad + centralidad
K    = 3
B    = 1000   # remuestras bootstrap (baja a 500 si va lento)

base_df = m[m['tipologia_label'].isin(['Espontáneo','Planificado'])].dropna(subset=[OUT,'espontaneo']+CTRL).copy()
print(f'N: {len(base_df)}  Esp={int(base_df.espontaneo.sum())}  Plan={int((base_df.espontaneo==0).sum())}')

def match_att(d, return_match=False):
    Xs = StandardScaler().fit_transform(d[CTRL].values)
    d  = d.assign(ps=LogisticRegression(max_iter=1000, random_state=42).fit(Xs, d['espontaneo'].values).predict_proba(Xs)[:,1])
    cal = 0.2*d['ps'].std()
    it = d.index[d.espontaneo==1]; ic = d.index[d.espontaneo==0]
    if len(ic) < K: return (np.nan,[],[],d) if return_match else np.nan
    dist, nn = cKDTree(d.loc[ic,'ps'].values.reshape(-1,1)).query(d.loc[it,'ps'].values.reshape(-1,1), k=K)
    if K == 1: dist, nn = dist.reshape(-1,1), nn.reshape(-1,1)
    yt = d.loc[it,OUT].values; yc = d.loc[ic,OUT].values
    mask = dist <= cal; cnt = mask.sum(1); keep = cnt > 0
    ctrl_mean = np.where(mask, yc[nn], 0.0).sum(1)/np.maximum(cnt,1)
    att = (yt - ctrl_mean)[keep].mean() if keep.any() else np.nan
    if not return_match: return att
    itm = it[keep].tolist(); icv = ic.values; icm = []
    for i in np.where(keep)[0]: icm.extend(icv[nn[i][mask[i]]].tolist())
    return att, itm, icm, d

# Punto estimado + matching (para balance)
att, itm, icm, d_ps = match_att(base_df, return_match=True)

# Bootstrap honesto: re-estima PS y re-empareja en cada remuestra
rng = np.random.default_rng(42); boot = []
for _ in range(B):
    a = match_att(base_df.sample(frac=1, replace=True, random_state=int(rng.integers(1e9))).reset_index(drop=True))
    if np.isfinite(a): boot.append(a)
boot = np.array(boot)
ci = (np.percentile(boot,2.5), np.percentile(boot,97.5))
p_boot = 2*min((boot<=0).mean(), (boot>=0).mean())
sig = lambda p:'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
print(f'\nPSM k={K} (con reemplazo, outcome={OUT}):')
print(f'  Espontáneos emparejados: {len(itm)} de {int(base_df.espontaneo.sum())}')
print(f'  ATT = {att:+.4f}   IC95_boot=[{ci[0]:+.4f}, {ci[1]:+.4f}]   p_boot={p_boot:.4f} {sig(p_boot)}   (B={len(boot)})')

# Balance
def smd(a,b):
    s=np.sqrt((a.std()**2+b.std()**2)/2); return abs(a.mean()-b.mean())/s if s>0 else 0
print('\n  Balance SMD (pre → post):')
for v in CTRL+['ps']:
    pre  = smd(d_ps[d_ps.espontaneo==1][v], d_ps[d_ps.espontaneo==0][v])
    post = smd(d_ps.loc[itm,v], d_ps.loc[icm,v])
    print(f'    {v:<16} {pre:.3f} → {post:.3f}')
att_psm, p_psm, ci_psm = att, p_boot, ci

In [ ]:
OUT  = 'vitalidad_vecinal'
CTRL = ['log_renta','log_densidad','dist_centro_km','log_bc_mean','log_airbnb']   # + turismo
K    = 2                                                                          # 2 vecinos

df = m[m['tipologia_label'].isin(['Espontáneo','Planificado'])].dropna(subset=[OUT,'espontaneo']+CTRL).copy()
print(f'N: {len(df)}  Esp={int(df.espontaneo.sum())}  Plan={int((df.espontaneo==0).sum())}')

sc = StandardScaler(); Xs = sc.fit_transform(df[CTRL].values); T = df['espontaneo'].values
lp = LogisticRegression(max_iter=1000, random_state=42).fit(Xs, T)
df['ps'] = lp.predict_proba(Xs)[:,1]

CAL = 0.2*df['ps'].std()
it = df[df.espontaneo==1].index.values; ic = df[df.espontaneo==0].index.values
pt = df.loc[it,'ps'].values; pc = df.loc[ic,'ps'].values
tree = cKDTree(pc.reshape(-1,1))
dist, nn = tree.query(pt.reshape(-1,1), k=K)
if K == 1: dist, nn = dist.reshape(-1,1), nn.reshape(-1,1)

diffs, itm, icm = [], [], []
for i in range(len(it)):
    sel = nn[i][dist[i] <= CAL]
    if len(sel) == 0: continue
    cids = ic[sel]
    diffs.append(df.loc[it[i], OUT] - df.loc[cids, OUT].mean())
    itm.append(it[i]); icm.extend(cids.tolist())
diffs = np.array(diffs)
att = diffs.mean(); se = diffs.std(ddof=1)/np.sqrt(len(diffs))
patt = 2*(1 - stats.t.cdf(abs(att/se), df=len(diffs)-1))
ci = (att - 1.96*se, att + 1.96*se)
sig = lambda p:'***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'n.s.'
print(f'\nPSM k={K} (con reemplazo, +turismo, outcome={OUT}):')
print(f'  Espontáneos emparejados: {len(diffs)} de {len(it)}')
print(f'  ATT = {att:+.4f}  {sig(patt)}  p={patt:.4f}  IC95=[{ci[0]:.4f},{ci[1]:.4f}]')
print('  (p naive optimista por reemplazo; para p honesta usa la celda con bootstrap)')

# Balance — MIRA si densidad/centralidad siguen descuadrados
def smd(a,b):
    s=np.sqrt((a.std()**2+b.std()**2)/2); return abs(a.mean()-b.mean())/s if s>0 else 0
print('\n  Balance SMD (pre → post):')
for v in CTRL+['ps']:
    pre  = smd(df[df.espontaneo==1][v], df[df.espontaneo==0][v])
    post = smd(df.loc[itm,v], df.loc[icm,v])
    flag = '  <-- mal (>0.1)' if post>0.1 else ''
    print(f'    {v:<16} {pre:.3f} → {post:.3f}{flag}')
att_psm, p_psm, ci_psm = att, patt, ci

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from scipy.stats import gaussian_kde

BASE = r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\Vitalidad'
m = pd.read_csv(BASE + r'\madrid_vitalidad_h3_master_v5.csv')
CTRL = ['log_renta','log_densidad','dist_centro_km','log_bc_mean']     # renta + densidad + centralidad
df = m[m['tipologia_label'].isin(['Espontáneo','Planificado'])].dropna(subset=['espontaneo']+CTRL).copy()

Xs = StandardScaler().fit_transform(df[CTRL].values)
df['ps'] = LogisticRegression(max_iter=1000, random_state=42).fit(Xs, df['espontaneo'].values).predict_proba(Xs)[:,1]

pt = df.loc[df.espontaneo==1,'ps'].values    # Espontáneo (participantes)
pc = df.loc[df.espontaneo==0,'ps'].values    # Planificado (no participantes)
lo, hi = max(pt.min(), pc.min()), min(pt.max(), pc.max())

xs = np.linspace(0, 1, 200)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(xs, gaussian_kde(pc)(xs), color='#1f77b4', lw=2, label='Planificado (no participantes)')
ax.plot(xs, gaussian_kde(pt)(xs), color='#d62728', lw=2, label='Espontáneo (participantes)')
ax.axvspan(lo, hi, color='grey', alpha=0.10, label='Región de soporte común')
ax.set_xlabel('Propensity score · P(espontáneo | renta, densidad, centralidad)')
ax.set_ylabel('Densidad')
ax.set_title('Soporte común — distribución del propensity score')
ax.legend(); plt.tight_layout(); plt.show()

print(f'Soporte común: [{lo:.3f}, {hi:.3f}]  |  fuera: {((pt<lo)|(pt>hi)).sum()} esp, {((pc<lo)|(pc>hi)).sum()} plan')

## 13. MÉTODO B — Frontera / Diff-in-discontinuities

Comparación local entre hexágonos H3 adyacentes Espontáneo↔Planificado.
Tres refinamientos sobre v3:
1. **Caliper de renta**: sólo pares con |Δlog_renta| ≤ 0.20 (comparabilidad local → supuesto RDD).
2. **Within-pair FE** con controles, incluyendo touristificación.
3. **Heterogeneidad centro/periferia**: ¿la morfología importa más donde el casco de pueblo
   limita con PAUs contemporáneos (periferia) que donde el casco viejo limita con el Ensanche (centro)?

In [ ]:
# Identificar pares adyacentes
ge = gdf[gdf.tipologia_label=='Espontáneo'][['hex_id','geometry']].copy()
gp = gdf[gdf.tipologia_label=='Planificado'][['hex_id','geometry']].copy()
geb = ge.copy(); geb['geometry']=ge.geometry.buffer(2)
adj = gpd.sjoin(geb.rename(columns={'hex_id':'hex_esp'}),
                gp.rename(columns={'hex_id':'hex_plan'}),
                how='inner', predicate='intersects')
pairs = adj[['hex_esp','hex_plan']].drop_duplicates().reset_index(drop=True)
print(f'Pares adyacentes Esp-Plan: {len(pairs):,}')

# Variables a ambos lados
cols = ['hex_id',OUT,'dim1','dim2','dim3','log_renta','log_densidad','dist_centro_km',
        'log_bc_mean','log_airbnb','barrio_km2','pct_barrio']
cols = [c for c in cols if c in m.columns]
mvv = m[cols].copy()
pf = (pairs
  .merge(mvv.add_suffix('_esp').rename(columns={'hex_id_esp':'hex_esp'}), on='hex_esp', how='left')
  .merge(mvv.add_suffix('_plan').rename(columns={'hex_id_plan':'hex_plan'}), on='hex_plan', how='left'))

# Filtrar pares con datos completos en outcome y controles
need = [f'{OUT}_esp',f'{OUT}_plan','log_renta_esp','log_renta_plan',
        'log_densidad_esp','log_densidad_plan','dist_centro_km_esp','dist_centro_km_plan',
        'log_bc_mean_esp','log_bc_mean_plan','log_airbnb_esp','log_airbnb_plan']
pf = pf.dropna(subset=need).copy()
pf['d_renta'] = pf['log_renta_esp'] - pf['log_renta_plan']
print(f'Pares con datos completos: {len(pf):,}')
print(f'|Δlog_renta| mediana: {pf["d_renta"].abs().median():.3f}')

In [ ]:
# ── B1: Within-pair FE — muestra completa ───────────────────────────────────
def build_panel(pframe):
    pframe = pframe.reset_index(drop=True).copy()
    pframe['pair_id'] = range(len(pframe))
    recs=[]
    for _,r in pframe.iterrows():
        for side,T in [('esp',1),('plan',0)]:
            recs.append({'pair_id':r['pair_id'],'T':T,'Y':r[f'{OUT}_{side}'],
                'log_renta':r[f'log_renta_{side}'],'log_densidad':r[f'log_densidad_{side}'],
                'dist_centro_km':r[f'dist_centro_km_{side}'],'log_bc_mean':r[f'log_bc_mean_{side}'],
                'log_airbnb':r[f'log_airbnb_{side}']})
    return pd.DataFrame(recs)

panel = build_panel(pf)
mfe = smf.ols('Y ~ T + log_renta + log_densidad + dist_centro_km + log_bc_mean + log_airbnb + C(pair_id)',
              data=panel).fit(cov_type='HC3')
bT, pT = mfe.params['T'], mfe.pvalues['T']; ciT = mfe.conf_int().loc['T']
print('=== B1: Within-pair FE (muestra completa) ===')
print(f'  N pares={len(pf)}  beta_T={bT:+.4f}  {sig(pT)}  p={pT:.4f}  IC95=[{ciT[0]:.4f},{ciT[1]:.4f}]')

# ── B2: Caliper de renta (|Δlog_renta| ≤ 0.20) ───────────────────────────────
pf_cal = pf[pf['d_renta'].abs() <= 0.20].copy()
panel_cal = build_panel(pf_cal)
mfe_cal = smf.ols('Y ~ T + log_renta + log_densidad + dist_centro_km + log_bc_mean + log_airbnb + C(pair_id)',
                  data=panel_cal).fit(cov_type='HC3')
bTc, pTc = mfe_cal.params['T'], mfe_cal.pvalues['T']; ciTc = mfe_cal.conf_int().loc['T']
print(f'\n=== B2: Within-pair FE + caliper renta (|Δlog_renta|≤0.20) ===')
print(f'  N pares={len(pf_cal)}  beta_T={bTc:+.4f}  {sig(pTc)}  p={pTc:.4f}  IC95=[{ciTc[0]:.4f},{ciTc[1]:.4f}]')

In [ ]:
# ── B3: Heterogeneidad CENTRO / PERIFERIA ────────────────────────────────────
# Periférico: distancia al centro del par > mediana global de hexágonos espontáneos
dist_med = m[m.tipologia_label=='Espontáneo']['dist_centro_km'].median()
pf['perif'] = (pf['dist_centro_km_esp'] > dist_med).astype(int)
print(f'Umbral periferia (mediana dist_centro Esp): {dist_med:.2f} km')
print(f'Pares centrales: {(pf.perif==0).sum()}  |  periféricos: {(pf.perif==1).sum()}')

panel_h = build_panel(pf)
# Reañadir perif al panel (constante por par)
perif_map = dict(zip(range(len(pf)), pf['perif'].values))
panel_h['perif'] = panel_h['pair_id'].map(perif_map)
panel_h['T_perif'] = panel_h['T'] * panel_h['perif']

mfe_h = smf.ols('Y ~ T + T_perif + log_renta + log_densidad + dist_centro_km + log_bc_mean + log_airbnb + C(pair_id)',
                data=panel_h).fit(cov_type='HC3')
b_centro = mfe_h.params['T']           # efecto morfológico en pares centrales
p_centro = mfe_h.pvalues['T']
b_inter  = mfe_h.params['T_perif']     # efecto adicional en periferia
p_inter  = mfe_h.pvalues['T_perif']
b_perif  = b_centro + b_inter          # efecto morfológico en periferia
print('\n=== B3: Heterogeneidad centro/periferia (within-pair FE) ===')
print(f'  Efecto en CENTRO  (casco viejo ↔ Ensanche):  beta={b_centro:+.4f}  {sig(p_centro)}  p={p_centro:.4f}')
print(f'  Interacción periferia:                        beta={b_inter:+.4f}  {sig(p_inter)}  p={p_inter:.4f}')
print(f'  Efecto en PERIFERIA (casco pueblo ↔ PAU):    beta={b_perif:+.4f}')

# Test separado por submuestra (más transparente)
print('\n  --- Submuestras separadas ---')
for nombre, sub in [('CENTRO', pf[pf.perif==0]), ('PERIFERIA', pf[pf.perif==1])]:
    pn = build_panel(sub)
    mm = smf.ols('Y ~ T + log_renta + log_densidad + dist_centro_km + log_bc_mean + log_airbnb + C(pair_id)',
                 data=pn).fit(cov_type='HC3')
    bb,pp = mm.params['T'], mm.pvalues['T']; cc=mm.conf_int().loc['T']
    print(f'  {nombre:<10} N={len(sub):>4}  beta_T={bb:+.4f}  {sig(pp)}  p={pp:.4f}  IC95=[{cc[0]:.4f},{cc[1]:.4f}]')

In [ ]:
# ── B4: % de pares donde Esp > Plan, por zona y dimensión ───────────────────
fig, axes = plt.subplots(1,3, figsize=(18,5))

# Distribución del diferencial outcome
pf['d_out'] = pf[f'{OUT}_esp'] - pf[f'{OUT}_plan']
for nombre,sub,c in [('Centro',pf[pf.perif==0],'#2b83ba'),('Periferia',pf[pf.perif==1],'#d7191c')]:
    axes[0].hist(sub['d_out'], bins=30, alpha=0.55, color=c, label=nombre, density=True)
axes[0].axvline(0, color='black', ls='--', lw=1)
axes[0].set_xlabel('Vitalidad vecinal: Esp − Plan'); axes[0].legend()
axes[0].set_title('Diferencial en frontera\npor zona')

# % Esp > Plan por zona
zonas = {'Centro':pf[pf.perif==0], 'Periferia':pf[pf.perif==1], 'Todos':pf}
pcts = {k:(v['d_out']>0).mean()*100 for k,v in zonas.items()}
bars=axes[1].bar(pcts.keys(), pcts.values(),
    color=['#2b83ba','#d7191c','#888'], alpha=0.8)
axes[1].axhline(50, color='black', ls='--', lw=1, label='50% (H0)')
axes[1].set_ylabel('% pares Esp > Plan'); axes[1].set_ylim(0,100); axes[1].legend()
axes[1].set_title('¿El espontáneo supera\nal planificado vecino?')
for b,v in zip(bars,pcts.values()):
    axes[1].text(b.get_x()+b.get_width()/2, v+1.5, f'{v:.1f}%', ha='center', fontweight='bold')

# Efecto por zona (coef plot)
zona_lab=['Centro\n(↔Ensanche)','Periferia\n(↔PAU)']
zona_b=[]; zona_ci=[]
for sub in [pf[pf.perif==0], pf[pf.perif==1]]:
    pn=build_panel(sub)
    mm=smf.ols('Y ~ T + log_renta + log_densidad + dist_centro_km + log_bc_mean + log_airbnb + C(pair_id)',
               data=pn).fit(cov_type='HC3')
    zona_b.append(mm.params['T']); zona_ci.append(mm.conf_int().loc['T'])
yp=range(len(zona_b))
axes[2].errorbar(zona_b, yp,
    xerr=[[zona_b[i]-zona_ci[i][0] for i in range(len(zona_b))],
          [zona_ci[i][1]-zona_b[i] for i in range(len(zona_b))]],
    fmt='o', color='#333', capsize=5, markersize=8)
axes[2].axvline(0, color='red', ls='--', lw=1)
axes[2].set_yticks(list(yp)); axes[2].set_yticklabels(zona_lab)
axes[2].set_xlabel('Efecto morfológico (β_T within-pair)')
axes[2].set_title('Efecto espontáneo por zona\n(con IC95%)')
plt.tight_layout(); plt.savefig('imagenes_v4/frontera_v4.png', dpi=180, bbox_inches='tight'); plt.show()

## 14. Resumen de resultados

In [ ]:
print('='*72)
print('  VITALIDAD VECINAL v4 — Espontáneo vs Planificado')
print('='*72)
da=m[m.tipologia_label=='Espontáneo']['vitalidad_vecinal'].dropna()
db=m[m.tipologia_label=='Planificado']['vitalidad_vecinal'].dropna()
_,pd_=stats.ttest_ind(da,db); dd=(da.mean()-db.mean())/np.sqrt((da.std()**2+db.std()**2)/2)
print(f'\nDescriptivo bruto: Esp={da.mean():.3f}  Plan={db.mean():.3f}  d={dd:+.3f}  {sig(pd_)}')
print()
print('--- MÉTODO A: PSM (control touristificación) ---')
print(f'  ATT = {att_psm:+.4f}  {sig(p_psm)}  IC95=[{ci_psm[0]:.4f},{ci_psm[1]:.4f}]')
print()
print('--- MÉTODO B: Frontera / Diff-in-disc ---')
print(f'  B1 within-pair FE (completo):   beta_T={bT:+.4f}  {sig(pT)}')
print(f'  B2 + caliper renta:             beta_T={bTc:+.4f}  {sig(pTc)}')
print(f'  B3 CENTRO (↔Ensanche):          beta={b_centro:+.4f}  {sig(p_centro)}')
print(f'     PERIFERIA (↔PAU):            beta={b_perif:+.4f}')
print(f'     interacción periferia:       beta={b_inter:+.4f}  {sig(p_inter)}')
print()
print('--- INTERPRETACIÓN ---')
print(f'  Validación: correlación v4-Airbnb baja/negativa (vs +0.555 en v2).')
print(f'  El índice ya mide vida vecinal, no animación turística.')
print('='*72)

In [ ]:
# Exportar master v4
expc = ['hex_id','tipologia_label','morfologia_continua','espontaneo','dist_centro_km',
        'vitalidad_vecinal','dim1','dim2','dim3',
        'barrio_km2','pct_barrio','pct_host','div_actividad',
        'terrazas_ext_km2','n_parques_500m','dist_mercado_km','n_mercados_500m','dist_parque_km',
        'airbnb_ent_km2','vut_km2','renta_persona','densidad_pob_km2',
        'log_renta','log_densidad','log_bc_mean','log_airbnb','ps']
if 'ps' in df.columns:
    m_exp = m.merge(df[['hex_id','ps']].drop_duplicates(), on='hex_id', how='left')
else:
    m_exp = m.copy()
valid=[c for c in expc if c in m_exp.columns]
m_exp[valid].to_csv(VITALIDAD/'madrid_vitalidad_h3_master_v4_limpio.csv', index=False)
print(f'Exportado: madrid_vitalidad_h3_master_v4_limpio.csv ({len(m_exp)} x {len(valid)})')